# 第4章 探索与利用

## 目录1. [环境搭建](#1-环境搭建)2. [探索与利用的权衡问题](#2-探索与利用的权衡问题)3. [多臂老虎机](#3-多臂老虎机)4. [ε-贪心策略](#4-ε-贪心策略)5. [UCB算法](#5-UCB算法)6. [Thompson采样算法](#6-Thompson采样算法)7. [编程题](#7-编程题)

---

## 1. 环境搭建

首先，我们安装强化学习所需的依赖库。

In [ ]:
# 安装强化学习核心依赖库
!pip install numpy matplotlib gym pygame torch torchvision scipy -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random
from scipy.stats import beta as beta_dist

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("环境搭建完成！")
print(f"NumPy版本: {np.__version__}")

---

## 2. 探索与利用的权衡问题

### 2.1 探索与利用的定义

**探索与利用的权衡（Exploration-Exploitation Trade-off）**是强化学习永恒的核心挑战。

- **利用（Exploitation）**：选择当前估计价值最高的动作，最大化当前即时奖励
- **探索（Exploration）**：选择当前估计价值不是最高的动作，获取更多环境信息

In [ ]:
def visualize_explore_exploit():
    """可视化探索与利用的权衡"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左图：不同epsilon下的收敛效果
    epsilons = [0.0, 0.1, 0.3, 0.5]
    colors = ['blue', 'orange', 'green', 'red']
    
    for eps, color in zip(epsilons, colors):
        rewards = []
        q_values = [0, 0, 0, 0]  # 4个动作的Q值
        
        for episode in range(100):
            if random.random() < eps:
                action = random.randint(0, 3)
            else:
                action = np.argmax(q_values)
            
            reward = np.random.normal([10, 5, 3, 2][action], 1)
            q_values[action] += 0.1 * (reward - q_values[action])
            rewards.append(np.mean(q_values))
        
        axes[0].plot(rewards, label=f'ε={eps}', color=color, linewidth=2)
    
    axes[0].set_xlabel('回合数', fontsize=12)
    axes[0].set_ylabel('平均奖励', fontsize=12)
    axes[0].set_title('探索率ε对学习效果的影响', fontsize=14)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 右图：探索vs利用示意图
    x = np.linspace(0, 1, 100)
    axes[1].fill_between(x, 0, 1-x, alpha=0.3, label='利用区域', color='blue')
    axes[1].fill_between(x, 1-x, 1, alpha=0.3, label='探索区域', color='orange')
    axes[1].plot(x, 1-x, 'b-', linewidth=2)
    axes[1].set_xlabel('探索率ε', fontsize=12)
    axes[1].set_ylabel('利用率 (1-ε)', fontsize=12)
    axes[1].set_title('探索与利用的权衡', fontsize=14)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('explore_exploit.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_explore_exploit()

---

## 3. 多臂老虎机

### 3.1 多臂老虎机问题形式化

多臂老虎机问题（Multi-Armed Bandit Problem, MAB）是探索与利用权衡的经典场景。

**问题定义**：
- 有K台老虎机（臂）
- 每台老虎机a有未知的期望奖励分布$\mu_a$
- 进行T次拉臂操作
- 目标：最大化T次拉臂的总期望奖励

In [ ]:
class MultiArmedBandit:
    """多臂老虎机环境"""
    def __init__(self, n_arms=10, true_probs=None):
        self.n_arms = n_arms
        if true_probs is None:
            self.true_probs = np.random.rand(n_arms)  # 真实奖励概率
        else:
            self.true_probs = true_probs
        self.counts = np.zeros(n_arms)  # 每个臂的拉取次数
        self.values = np.zeros(n_arms)  # 每个臂的估计价值
    
    def pull(self, arm):
        """拉取指定臂"""
        reward = 1 if np.random.random() < self.true_probs[arm] else 0
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] = self.values[arm] + (reward - self.values[arm]) / n
        return reward
    
    def get_estimated_values(self):
        """获取估计价值"""
        return self.values.copy()
    
    def get_counts(self):
        """获取拉取次数"""
        return self.counts.copy()

# 测试多臂老虎机
print("测试多臂老虎机...")
bandit = MultiArmedBandit(n_arms=5, true_probs=[0.3, 0.5, 0.7, 0.4, 0.6])
print(f"真实奖励概率: {bandit.true_probs}")
print(f"最优臂: {np.argmax(bandit.true_probs)}")

for i in range(20):
    arm = np.random.randint(5)
    reward = bandit.pull(arm)
    if i < 5:
        print(f"第{i+1}次: 拉取臂{arm}, 奖励={reward}")

print(f"估计价值: {bandit.get_estimated_values()}")

---

## 4. ε-贪心策略

### 4.1 ε-贪心策略定义

**ε-贪心策略（ε-Greedy Policy）**是最简单、最常用的探索策略：
- 以概率ε进行探索（随机选择）
- 以概率1-ε进行利用（选择当前估计价值最高的臂）

In [ ]:
class EpsilonGreedy:
    """ε-贪心策略"""
    def __init__(self, n_arms, epsilon=0.1):
        self.n_arms = n_arms
        self.epsilon = epsilon
        self.values = np.zeros(n_arms)
        self.counts = np.zeros(n_arms)
    
    def select_action(self):
        """选择动作"""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_arms)
        else:
            return np.argmax(self.values)
    
    def update(self, action, reward):
        """更新Q值"""
        self.counts[action] += 1
        n = self.counts[action]
        self.values[action] = self.values[action] + (reward - self.values[action]) / n

# 测试ε-贪心策略
print("测试ε-贪心策略...")
agent = EpsilonGreedy(n_arms=5, epsilon=0.1)
true_probs = [0.3, 0.5, 0.7, 0.4, 0.6]

for i in range(100):
    action = agent.select_action()
    reward = 1 if np.random.random() < true_probs[action] else 0
    agent.update(action, reward)

print(f"估计价值: {agent.values}")
print(f"真实概率: {true_probs}")
print(f"选择的最优动作: {np.argmax(agent.values)}")

### 4.2 衰减ε-贪心策略

**衰减ε-贪心策略**随着时间步增加逐渐降低探索概率，常见的衰减函数：
- 平方根衰减：$\epsilon_t = \frac{c}{\sqrt{t}}$
- 对数衰减：$\epsilon_t = \frac{c}{\ln(t+1)}$
- 指数衰减：$\epsilon_t = \epsilon_0 \cdot \alpha^t$

In [ ]:
class DecayingEpsilonGreedy:
    """衰减ε-贪心策略"""
    def __init__(self, n_arms, decay_type='sqrt', c=1.0, epsilon0=1.0, alpha=0.99):
        self.n_arms = n_arms
        self.decay_type = decay_type
        self.c = c
        self.epsilon0 = epsilon0
        self.alpha = alpha
        self.values = np.zeros(n_arms)
        self.counts = np.zeros(n_arms)
        self.t = 0
    
    def get_epsilon(self):
        """计算当前epsilon值"""
        if self.decay_type == 'sqrt':
            return self.c / np.sqrt(self.t + 1)
        elif self.decay_type == 'log':
            return self.c / np.log(self.t + 2)
        elif self.decay_type == 'exp':
            return self.epsilon0 * (self.alpha ** self.t)
        return self.c / np.sqrt(self.t + 1)
    
    def select_action(self):
        """选择动作"""
        epsilon = self.get_epsilon()
        if np.random.random() < epsilon:
            return np.random.randint(self.n_arms)
        else:
            return np.argmax(self.values)
    
    def update(self, action, reward):
        """更新Q值"""
        self.t += 1
        self.counts[action] += 1
        n = self.counts[action]
        self.values[action] = self.values[action] + (reward - self.values[action]) / n

# 对比不同衰减策略
def compare_decay_strategies(n_arms=10, n_steps=1000, n_runs=50):
    """对比不同衰减策略"""
    decay_types = ['sqrt', 'log', 'exp']
    colors = ['blue', 'orange', 'green']
    
    plt.figure(figsize=(12, 5))
    
    for decay_type, color in zip(decay_types, colors):
        total_rewards_list = []
        
        for run in range(n_runs):
            agent = DecayingEpsilonGreedy(n_arms, decay_type=decay_type)
            true_probs = np.random.rand(n_arms)
            total_rewards = []
            cumulative = 0
            
            for step in range(n_steps):
                action = agent.select_action()
                reward = 1 if np.random.random() < true_probs[action] else 0
                agent.update(action, reward)
                cumulative += reward
                total_rewards.append(cumulative)
            
            total_rewards_list.append(total_rewards)
        
        avg_rewards = np.mean(total_rewards_list, axis=0)
        plt.plot(avg_rewards, label=decay_type, color=color, linewidth=2)
    
    plt.xlabel('时间步', fontsize=12)
    plt.ylabel('累积奖励', fontsize=12)
    plt.title('不同ε衰减策略对比', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('decaying_epsilon.png', dpi=150)
    plt.show()

compare_decay_strategies()

---

## 5. UCB算法

### 5.1 UCB1算法定义

**上置信界算法（UCB, Upper Confidence Bound）**是一种基于不确定性的探索策略：
- 不仅考虑当前估计价值，还考虑不确定性
- 优先探索"估计价值较高但不确定性较大"的臂

UCB1算法的数学定义：
$$a_{UCB1} = \arg\max_{a \in \mathcal{A}} \left( \hat{\mu}_a + c \cdot \sqrt{\frac{2 \ln t}{N_a}} \right)$$

其中：
- $\hat{\mu}_a$是臂a的当前估计价值
- $c$是探索系数
- $t$是总拉臂次数
- $N_a$是臂a的已拉臂次数

In [ ]:
class UCB1:
    """UCB1算法"""
    def __init__(self, n_arms, c=2.0):
        self.n_arms = n_arms
        self.c = c
        self.values = np.zeros(n_arms)
        self.counts = np.zeros(n_arms)
        self.total_counts = 0
    
    def select_action(self):
        """选择动作"""
        # 初始探索：确保每个臂都被拉过一次
        if self.total_counts < self.n_arms:
            return self.total_counts
        
        # 计算UCB值
        ucb_values = self.values + self.c * np.sqrt(2 * np.log(self.total_counts) / (self.counts + 1e-6))
        return np.argmax(ucb_values)
    
    def update(self, action, reward):
        """更新Q值"""
        self.total_counts += 1
        self.counts[action] += 1
        n = self.counts[action]
        self.values[action] = self.values[action] + (reward - self.values[action]) / n

# 测试UCB1
print("测试UCB1算法...")
agent = UCB1(n_arms=5, c=2.0)
true_probs = [0.3, 0.5, 0.7, 0.4, 0.6]

for i in range(100):
    action = agent.select_action()
    reward = 1 if np.random.random() < true_probs[action] else 0
    agent.update(action, reward)

print(f"估计价值: {agent.values}")
print(f"真实概率: {true_probs}")
print(f"选择的最优动作: {np.argmax(agent.values)}")

### 5.2 UCB2算法

**UCB2算法**是对UCB1的改进，引入指数衰减的探索项调节：
$$a_{UCB2} = \arg\max_{a \in \mathcal{A}} \left( \hat{\mu}_a + c \cdot \sqrt{\frac{2 \ln t}{N_a}} \cdot \alpha^{N_a} \right)$$

其中$\alpha \in (0,1)$是指数衰减系数。

In [ ]:
class UCB2:
    """UCB2算法"""
    def __init__(self, n_arms, c=2.0, alpha=0.5):
        self.n_arms = n_arms
        self.c = c
        self.alpha = alpha
        self.values = np.zeros(n_arms)
        self.counts = np.zeros(n_arms)
        self.total_counts = 0
    
    def select_action(self):
        """选择动作"""
        if self.total_counts < self.n_arms:
            return self.total_counts
        
        # 计算UCB2值（带指数衰减）
        exploration = self.c * np.sqrt(2 * np.log(self.total_counts) / (self.counts + 1e-6))
        decay_factor = np.array([self.alpha ** self.counts[i] for i in range(self.n_arms)])
        ucb_values = self.values + exploration * decay_factor
        return np.argmax(ucb_values)
    
    def update(self, action, reward):
        """更新Q值"""
        self.total_counts += 1
        self.counts[action] += 1
        n = self.counts[action]
        self.values[action] = self.values[action] + (reward - self.values[action]) / n

# 对比UCB1和UCB2
def compare_ucb(n_arms=10, n_steps=1000, n_runs=50):
    """对比UCB1和UCB2"""
    algorithms = [('UCB1', UCB1), ('UCB2', UCB2)]
    colors = ['blue', 'orange']
    
    plt.figure(figsize=(12, 5))
    
    for (name, Algo), color in zip(algorithms, colors):
        total_rewards_list = []
        
        for run in range(n_runs):
            agent = Algo(n_arms, c=2.0)
            true_probs = np.random.rand(n_arms)
            total_rewards = []
            cumulative = 0
            
            for step in range(n_steps):
                action = agent.select_action()
                reward = 1 if np.random.random() < true_probs[action] else 0
                agent.update(action, reward)
                cumulative += reward
                total_rewards.append(cumulative)
            
            total_rewards_list.append(total_rewards)
        
        avg_rewards = np.mean(total_rewards_list, axis=0)
        plt.plot(avg_rewards, label=name, color=color, linewidth=2)
    
    plt.xlabel('时间步', fontsize=12)
    plt.ylabel('累积奖励', fontsize=12)
    plt.title('UCB1 vs UCB2', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('ucb_comparison.png', dpi=150)
    plt.show()

compare_ucb()

---

## 6. Thompson采样算法

### 6.1 Thompson采样定义

**Thompson采样（Thompson Sampling）**是一种基于贝叶斯统计的探索策略：
- 为每个臂假设先验分布（如Beta分布）
- 每次拉臂前从后验分布中采样
- 选择采样值最大的臂

对于伯努利奖励分布，使用Beta-Bernoulli模型：
- 后验分布：$\text{Beta}(\alpha, \beta)$
- 更新：$\alpha \leftarrow \alpha + r$, $\ \beta \leftarrow \beta + (1-r)$

In [ ]:
class ThompsonSampling:
    """Thompson采样算法（Beta-Bernoulli）"""
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.alpha = np.ones(n_arms)  # 成功次数 + 1
        self.beta = np.ones(n_arms)   # 失败次数 + 1
    
    def select_action(self):
        """从后验分布中采样并选择动作"""
        samples = [np.random.beta(self.alpha[i], self.beta[i]) for i in range(self.n_arms)]
        return np.argmax(samples)
    
    def update(self, action, reward):
        """更新后验参数"""
        if reward == 1:
            self.alpha[action] += 1
        else:
            self.beta[action] += 1
    
    def get_posterior_mean(self):
        """获取后验均值"""
        return self.alpha / (self.alpha + self.beta)

# 测试Thompson采样
print("测试Thompson采样算法...")
agent = ThompsonSampling(n_arms=5)
true_probs = [0.3, 0.5, 0.7, 0.4, 0.6]

for i in range(100):
    action = agent.select_action()
    reward = 1 if np.random.random() < true_probs[action] else 0
    agent.update(action, reward)

print(f"后验均值: {agent.get_posterior_mean()}")
print(f"真实概率: {true_probs}")
print(f"选择的最优动作: {np.argmax(agent.get_posterior_mean())}")

### 6.2 对比三种探索策略

In [ ]:
def compare_all_algorithms(n_arms=10, n_steps=1000, n_runs=50):
    """对比所有探索策略"""
    algorithms = [
        ('ε-贪心(ε=0.1)', lambda n: EpsilonGreedy(n, epsilon=0.1)),
        ('ε-贪心(ε=0.3)', lambda n: EpsilonGreedy(n, epsilon=0.3)),
        ('UCB1', lambda n: UCB1(n, c=2.0)),
        ('Thompson采样', lambda n: ThompsonSampling(n))
    ]
    colors = ['blue', 'orange', 'green', 'red']
    
    plt.figure(figsize=(14, 6))
    
    for (name, algo_fn), color in zip(algorithms, colors):
        total_rewards_list = []
        
        for run in range(n_runs):
            agent = algo_fn(n_arms)
            true_probs = np.random.rand(n_arms)
            total_rewards = []
            cumulative = 0
            
            for step in range(n_steps):
                action = agent.select_action()
                reward = 1 if np.random.random() < true_probs[action] else 0
                agent.update(action, reward)
                cumulative += reward
                total_rewards.append(cumulative)
            
            total_rewards_list.append(total_rewards)
        
        avg_rewards = np.mean(total_rewards_list, axis=0)
        plt.plot(avg_rewards, label=name, color=color, linewidth=2)
    
    plt.xlabel('时间步', fontsize=12)
    plt.ylabel('累积奖励', fontsize=12)
    plt.title('探索策略对比实验', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('algorithm_comparison.png', dpi=150)
    plt.show()

compare_all_algorithms()

### 6.3 稀疏奖励场景下的Thompson采样

在稀疏奖励场景下（多数臂的奖励概率很低），Thompson采样的自适应探索优势更加明显。

In [ ]:
def sparse_reward_experiment(n_arms=10, n_steps=2000, n_runs=50):
    """稀疏奖励场景实验"""
    # 创建一个臂有很高的奖励概率，其他都很低
    true_probs = np.zeros(n_arms)
    true_probs[0] = 0.8  # 一个高奖励臂
    true_probs[1:] = np.random.uniform(0.01, 0.05, n_arms - 1)  # 其他都是低奖励
    
    algorithms = [
        ('ε-贪心(ε=0.1)', lambda n: EpsilonGreedy(n, epsilon=0.1)),
        ('Thompson采样', lambda n: ThompsonSampling(n))
    ]
    colors = ['blue', 'red']
    
    plt.figure(figsize=(12, 5))
    
    for (name, algo_fn), color in zip(algorithms, colors):
        regrets = []
        
        for run in range(n_runs):
            agent = algo_fn(n_arms)
            optimal_prob = np.max(true_probs)
            cumulative_regret = []
            regret = 0
            
            for step in range(n_steps):
                action = agent.select_action()
                reward = 1 if np.random.random() < true_probs[action] else 0
                agent.update(action, reward)
                regret += optimal_prob - true_probs[action]
                cumulative_regret.append(regret)
            
            regrets.append(cumulative_regret)
        
        avg_regret = np.mean(regrets, axis=0)
        plt.plot(avg_regret, label=name, color=color, linewidth=2)
    
    plt.xlabel('时间步', fontsize=12)
    plt.ylabel('累积Regret', fontsize=12)
    plt.title('��疏��励场景：Regret对比', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('sparse_reward.png', dpi=150)
    plt.show()

print("稀疏奖励场景实验...")
sparse_reward_experiment()

---

## 7. 编程题

### 编程题1

**实现ε-贪心策略、UCB1算法和Thompson采样算法，在多臂老虎机问题上进行对比实验，分析不同策略在不同ε值、奖励分布下的性能差异。**

In [ ]:
"""
编程题1：实现ε-贪心策略、UCB1算法和Thompson采样算法
在多臂老虎机问题上进行对比实验
"""

class Environment:
    """多臂老虎机环境"""
    def __init__(self, n_arms=10, reward_type='bernoulli'):
        self.n_arms = n_arms
        self.reward_type = reward_type
        if reward_type == 'bernoulli':
            self.true_probs = np.random.rand(n_arms)
        elif reward_type == 'gaussian':
            self.true_means = np.random.randn(n_arms)
            self.std = 1.0
    
    def get_reward(self, action):
        """获取奖励"""
        if self.reward_type == 'bernoulli':
            return 1 if np.random.random() < self.true_probs[action] else 0
        elif self.reward_type == 'gaussian':
            return np.random.normal(self.true_means[action], self.std)
    
    def get_optimal_action(self):
        """获取最优动作"""
        if self.reward_type == 'bernoulli':
            return np.argmax(self.true_probs)
        elif self.reward_type == 'gaussian':
            return np.argmax(self.true_means)

def run_experiment(AlgoClass, n_arms, n_steps, **kwargs):
    """运行实验"""
    env = Environment(n_arms)
    agent = AlgoClass(n_arms, **kwargs)
    
    total_rewards = []
    optimal_counts = []
    cumulative = 0
    optimal_action = env.get_optimal_action()
    
    for step in range(n_steps):
        action = agent.select_action()
        reward = env.get_reward(action)
        agent.update(action, reward)
        cumulative += reward
        total_rewards.append(cumulative)
        optimal_counts.append(1 if action == optimal_action else 0)
    
    return total_rewards, optimal_counts

# 对比伯努利奖励分布
print("="*50)
print("实验1：伯努利奖励分布对比")
print("="*50)

n_arms = 10
n_steps = 2000
n_runs = 30

algorithms = [
    ('ε-贪心(ε=0.05)', EpsilonGreedy, {'epsilon': 0.05}),
    ('ε-贪心(ε=0.1)', EpsilonGreedy, {'epsilon': 0.1}),
    ('ε-贪心(ε=0.3)', EpsilonGreedy, {'epsilon': 0.3}),
    ('UCB1(c=1)', UCB1, {'c': 1.0}),
    ('UCB1(c=2)', UCB1, {'c': 2.0}),
    ('Thompson采样', ThompsonSampling, {})
]

results = {}
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown']

plt.figure(figsize=(14, 6))

for i, (name, Algo, kwargs) in enumerate(algorithms):
    all_rewards = []
    all_optimal = []
    
    for run in range(n_runs):
        rewards, optimal = run_experiment(Algo, n_arms, n_steps, **kwargs)
        all_rewards.append(rewards)
        all_optimal.append(optimal)
    
    avg_rewards = np.mean(all_rewards, axis=0)
    avg_optimal = np.mean(all_optimal, axis=0)
    
    results[name] = {
        'final_reward': avg_rewards[-1],
        'optimal_rate': np.mean(avg_optimal[-100:])
    }
    
    plt.plot(avg_rewards, label=name, color=colors[i], linewidth=2)

plt.xlabel('时间步', fontsize=12)
plt.ylabel('平均累积奖励', fontsize=12)
plt.title('不同ε值、奖励分布下的性能对比（伯努利）', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('programming_exercise_1.png', dpi=150)
plt.show()

print("\n结果汇总：")
for name, result in results.items():
    print(f"  {name}: 最终奖励={result['final_reward']:.2f}, 最优选择率={result['optimal_rate']:.2%}")

### 编程题2

**修改UCB算法，实现UCB2算法（带指数项调整的UCB变体），并与UCB1进行对比实验。**

In [ ]:
"""
编程题2：实现UCB2算法并与UCB1对比
"""

def ucb_comparison(n_arms=10, n_steps=2000, n_runs=30):
    """UCB1 vs UCB2对比"""
    algorithms = [
        ('UCB1(c=1)', UCB1, {'c': 1.0}),
        ('UCB1(c=2)', UCB1, {'c': 2.0}),
        ('UCB1(c=4)', UCB1, {'c': 4.0}),
        ('UCB2(α=0.3)', UCB2, {'c': 2.0, 'alpha': 0.3}),
        ('UCB2(α=0.5)', UCB2, {'c': 2.0, 'alpha': 0.5}),
        ('UCB2(α=0.7)', UCB2, {'c': 2.0, 'alpha': 0.7}),
    ]
    
    colors = ['blue', 'cyan', 'navy', 'red', 'orange', 'gold']
    
    plt.figure(figsize=(14, 6))
    
    for i, (name, Algo, kwargs) in enumerate(algorithms):
        all_rewards = []
        
        for run in range(n_runs):
            rewards, _ = run_experiment(Algo, n_arms, n_steps, **kwargs)
            all_rewards.append(rewards)
        
        avg_rewards = np.mean(all_rewards, axis=0)
        plt.plot(avg_rewards, label=name, color=colors[i], linewidth=2)
    
    plt.xlabel('时间步', fontsize=12)
    plt.ylabel('平均累积奖励', fontsize=12)
    plt.title('UCB1 vs UCB2：探索常数对比', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('programming_exercise_2.png', dpi=150)
    plt.show()

print("UCB1 vs UCB2对比实验完成")
ucb_comparison()

### 编程题3

**基于Beta分布实现Thompson采样算法，在稀疏奖励场景下与ε-贪心进行对比，分析Thompson采样的自适应探索优势。**

In [ ]:
"""
编程题3：稀疏奖励场景下的Thompson采样分析
"""

def sparse_reward_analysis(n_arms=20, n_steps=3000, n_runs=30):
    """稀疏奖励场景分析"""
    # 创建不同的稀疏奖励场景
    scenarios = [
        ('高稀疏', [0.9] + [0.02] * 19),
        ('中稀疏', [0.8] + [0.1] * 19),
        ('低稀疏', [0.6] + [0.3] * 19),
    ]
    
    plt.figure(figsize=(14, 10))
    
    for idx, (scenario_name, probs) in enumerate(scenarios):
        plt.subplot(2, 3, idx + 1)
        
        algorithms = [
            ('ε-贪心(0.1)', EpsilonGreedy, {'epsilon': 0.1}),
            ('ε-贪心(0.3)', EpsilonGreedy, {'epsilon': 0.3}),
            ('Thompson', ThompsonSampling, {})
        ]
        colors = ['blue', 'orange', 'red']
        
        for i, (name, Algo, kwargs) in enumerate(algorithms):
            regrets_list = []
            optimal_prob = max(probs)
            
            for run in range(n_runs):
                agent = Algo(n_arms, **kwargs)
                env = Environment(n_arms)
                env.true_probs = np.array(probs)
                
                regrets = []
                cumulative_regret = 0
                
                for step in range(n_steps):
                    action = agent.select_action()
                    reward = env.get_reward(action)
                    agent.update(action, reward)
                    cumulative_regret += optimal_prob - probs[action]
                    regrets.append(cumulative_regret)
                
                regrets_list.append(regrets)
            
            avg_regret = np.mean(regrets_list, axis=0)
            plt.plot(avg_regret, label=name, color=colors[i], linewidth=2)
        
        plt.xlabel('时间步', fontsize=10)
        plt.ylabel('累积Regret', fontsize=10)
        plt.title(f'{scenario_name}场景', fontsize=12)
        plt.legend(fontsize=8)
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('programming_exercise_3.png', dpi=150)
    plt.show()

print("稀疏奖励场景分析完成")
sparse_reward_analysis()

### 编程题4

**在头歌平台上完成"探索策略参数优化"任务，调试不同参数组合，分析参数对算法性能的影响。**

In [ ]:
"""
编程题4：参数优化分析
综合分析不同参数对算法性能的影响
"""

def parameter_tuning_analysis(n_arms=10, n_steps=2000, n_runs=20):
    """参数调优综合分析"""
    
    # 1. ε-贪心参数分析
    print("="*50)
    print("1. ε-贪心参数分析")
    print("="*50)
    
    epsilons = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]
    results_eg = {}
    
    for eps in epsilons:
        final_rewards = []
        for _ in range(n_runs):
            rewards, _ = run_experiment(EpsilonGreedy, n_arms, n_steps, epsilon=eps)
            final_rewards.append(rewards[-1])
        results_eg[eps] = np.mean(final_rewards)
        print(f"  ε={eps}: 累积奖励={results_eg[eps]:.2f}")
    
    # 2. UCB探索系数分析
    print("\n" + "="*50)
    print("2. UCB探索系数分析")
    print("="*50)
    
    cs = [0.5, 1.0, 2.0, 4.0, 8.0]
    results_ucb = {}
    
    for c in cs:
        final_rewards = []
        for _ in range(n_runs):
            rewards, _ = run_experiment(UCB1, n_arms, n_steps, c=c)
            final_rewards.append(rewards[-1])
        results_ucb[c] = np.mean(final_rewards)
        print(f"  c={c}: 累积奖励={results_ucb[c]:.2f}")
    
    # 3. 可视化
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(list(results_eg.keys()), list(results_eg.values()), 'bo-')
    axes[0].set_xlabel('ε值', fontsize=12)
    axes[0].set_ylabel('累积奖励', fontsize=12)
    axes[0].set_title('ε-贪心参数分析', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(list(results_ucb.keys()), list(results_ucb.values()), 'ro-')
    axes[1].set_xlabel('探索系数c', fontsize=12)
    axes[1].set_ylabel('累积奖励', fontsize=12)
    axes[1].set_title('UCB参数分析', fontsize=14)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('parameter_tuning.png', dpi=150)
    plt.show()
    
print("\n" + "="*50)
print("参数优化分析完成！")
print("="*50)
parameter_tuning_analysis()

In [ ]:
print("\n" + "="*50)
print("第4章 探索与利用 - 内容完成！")
print("="*50)